# Unsloth + OpenEnv GRPO (Autoscaling)

Minimal Colab notebook to train an autoscaling policy with **Unsloth + TRL GRPO** against the packaged OpenEnv environment at `envs/autoscale_env`.

In [ ]:
%%capture
!pip -q install --upgrade pip
!pip -q install unsloth trl transformers peft accelerate datasets fastapi uvicorn bitsandbytes xformers uv openenv-core

In [ ]:
import os
import subprocess
import time
from pathlib import Path

REPO_URL = "https://github.com/smadduri9/openenv-autoscale-rl"
REPO_DIR = Path("/content/OpenEnv2")
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print("Working dir:", REPO_DIR)

In [ ]:
trace_path = Path("traces.jsonl")
if trace_path.exists():
    print(f"[trace-check] Found existing trace file: {trace_path}")
else:
    print(f"[trace-check] Missing {trace_path}. Generating traces now...")
    !python3 generate_traces.py --num-traces 30 --output traces.jsonl --episode-length 120 --seed 7
    print(f"[trace-check] Generated trace file: {trace_path}")

print("[package] Validating packaged OpenEnv environment...")
!cd envs/autoscale_env && python3 -m uv lock && openenv validate --verbose

# Training cell uses packaged auto-launch path, so no manual server start is required here.
server_proc = None

In [ ]:
!python3 colab_train_rl.py \
  --base-url http://127.0.0.1:8000 \
  --auto-launch-server \
  --server-launch-mode packaged \
  --env-package-dir envs/autoscale_env \
  --trace-path traces.jsonl \
  --init-model unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit \
  --output-dir rl_model_unsloth_grpo \
  --num-prompts 48 \
  --prompt-seed 7 \
  --max-steps 30 \
  --learning-rate 5e-6 \
  --batch-size 1 \
  --gradient-accumulation-steps 2 \
  --num-generations 4 \
  --max-prompt-length 512 \
  --max-completion-length 8 \
  --seed 7

In [ ]:
!python3 eval_policy.py --trace-path traces.jsonl --max-traces 6 --sft-model-path sft_model_balanced --rl-model-path rl_model_unsloth_grpo

In [ ]:
from pathlib import Path
print("Saved RL outputs:")
for p in sorted(Path("rl_model_unsloth_grpo").glob("*")):
    print("-", p)

In [ ]:
if server_proc is not None:
    server_proc.terminate()
    print("Server terminated")
else:
    print("No manual server process to terminate (packaged auto-launch handles lifecycle).")